# 📊 E-Commerce Data Analytics

This notebook connects to the PostgreSQL database populated by the ETL pipeline and performs business analytics with visualizations.

**Prerequisites:**
- ETL pipeline has been run successfully (`python run_pipeline.py`)
- PostgreSQL is running with the `ecommerce_analytics` database

**Analyses Covered:**
1. Monthly Revenue Trend
2. Top 10 Selling Products
3. Revenue by Category
4. Top Customer Spending
5. Payment Method Distribution
6. Category Performance Heatmap

---
## 🔧 Setup & Database Connection

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import warnings

warnings.filterwarnings('ignore')

# Configure plot aesthetics
sns.set_theme(style='whitegrid', palette='husl', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.titleweight'] = 'bold'

# Database connection
# Update these settings to match your PostgreSQL configuration
DB_USER = 'postgres'
DB_PASSWORD = 'postgres'
DB_HOST = 'localhost'
DB_PORT = 5432
DB_NAME = 'ecommerce_analytics'

engine = create_engine(
    f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

print('✓ Connected to PostgreSQL database')

# Quick check: count rows in each table
for table in ['customers', 'products', 'orders']:
    count = pd.read_sql(f'SELECT COUNT(*) AS cnt FROM {table}', engine).iloc[0, 0]
    print(f'  → {table}: {count:,} rows')

---
## 📈 1. Monthly Revenue Trend

Track how revenue changes over time to identify seasonal patterns, growth trends, and anomalies.

In [ ]:
# Query: Monthly revenue trend
query_revenue_trend = """
SELECT
    order_year AS year,
    order_month AS month,
    COUNT(order_id) AS total_orders,
    SUM(revenue) AS total_revenue,
    ROUND(AVG(revenue)::numeric, 2) AS avg_order_value
FROM orders
GROUP BY order_year, order_month
ORDER BY order_year, order_month;
"""

df_revenue = pd.read_sql(query_revenue_trend, engine)

# Create a proper date column for plotting
df_revenue['period'] = pd.to_datetime(
    df_revenue['year'].astype(str) + '-' + df_revenue['month'].astype(str) + '-01'
)

print(f'Revenue data spans {len(df_revenue)} months')
df_revenue.head(10)

In [ ]:
# Visualization: Revenue Trend Line Chart
fig, ax1 = plt.subplots(figsize=(16, 7))

# Revenue line
color_revenue = '#2196F3'
ax1.plot(
    df_revenue['period'], df_revenue['total_revenue'],
    color=color_revenue, linewidth=2.5, marker='o', markersize=6,
    label='Total Revenue', zorder=3
)
ax1.fill_between(
    df_revenue['period'], df_revenue['total_revenue'],
    alpha=0.15, color=color_revenue
)
ax1.set_xlabel('Month', fontsize=13)
ax1.set_ylabel('Total Revenue (₹)', fontsize=13, color=color_revenue)
ax1.tick_params(axis='y', labelcolor=color_revenue)

# Order count on secondary axis
ax2 = ax1.twinx()
color_orders = '#FF9800'
ax2.bar(
    df_revenue['period'], df_revenue['total_orders'],
    alpha=0.3, color=color_orders, width=20, label='Order Count'
)
ax2.set_ylabel('Order Count', fontsize=13, color=color_orders)
ax2.tick_params(axis='y', labelcolor=color_orders)

plt.title('Monthly Revenue Trend & Order Volume', fontsize=18, fontweight='bold', pad=20)
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.92), fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved as revenue_trend.png')

---
## 🏆 2. Top 10 Selling Products

Identify the highest-revenue products to prioritize inventory and marketing spend.

In [ ]:
# Query: Top 10 products by revenue
query_top_products = """
SELECT
    p.product_name,
    p.category,
    SUM(o.quantity) AS total_qty,
    SUM(o.revenue) AS total_revenue,
    COUNT(o.order_id) AS order_count
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_name, p.category
ORDER BY total_revenue DESC
LIMIT 10;
"""

df_top_products = pd.read_sql(query_top_products, engine)
df_top_products

In [ ]:
# Visualization: Top Products Horizontal Bar Chart
fig, ax = plt.subplots(figsize=(14, 8))

colors = sns.color_palette('viridis', len(df_top_products))

bars = ax.barh(
    df_top_products['product_name'][::-1],
    df_top_products['total_revenue'][::-1],
    color=colors[::-1],
    edgecolor='white',
    linewidth=0.5
)

# Add value labels
for bar, rev in zip(bars, df_top_products['total_revenue'][::-1]):
    ax.text(
        bar.get_width() + bar.get_width() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'₹{rev:,.0f}',
        va='center', fontsize=10, fontweight='bold'
    )

ax.set_xlabel('Total Revenue (₹)', fontsize=13)
ax.set_title('Top 10 Products by Revenue', fontsize=18, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('top_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved as top_products.png')

---
## 🗂️ 3. Revenue by Category

Analyze how revenue is distributed across product categories to understand which segments drive the business.

In [ ]:
# Query: Revenue by category
query_category = """
SELECT
    p.category,
    COUNT(o.order_id) AS total_orders,
    SUM(o.quantity) AS total_qty,
    SUM(o.revenue) AS total_revenue,
    ROUND(AVG(o.revenue)::numeric, 2) AS avg_order_value,
    ROUND(SUM(o.revenue) * 100.0 / (SELECT SUM(revenue) FROM orders), 2) AS share_pct
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

df_category = pd.read_sql(query_category, engine)
df_category

In [ ]:
# Visualization: Category Revenue — Dual Chart (Bar + Pie)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Bar chart
palette = sns.color_palette('Set2', len(df_category))
bars = ax1.bar(
    range(len(df_category)), df_category['total_revenue'],
    color=palette, edgecolor='white', linewidth=0.8
)
ax1.set_xticks(range(len(df_category)))
ax1.set_xticklabels(df_category['category'], rotation=45, ha='right', fontsize=10)
ax1.set_ylabel('Total Revenue (₹)', fontsize=13)
ax1.set_title('Revenue by Category', fontsize=16, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2., height + height * 0.01,
        f'₹{height:,.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold'
    )

# Pie chart
wedges, texts, autotexts = ax2.pie(
    df_category['total_revenue'],
    labels=df_category['category'],
    autopct='%1.1f%%',
    colors=palette,
    startangle=140,
    pctdistance=0.85,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2)
)
ax2.set_title('Revenue Share (%)', fontsize=16, fontweight='bold')

for text in autotexts:
    text.set_fontsize(8)
    text.set_fontweight('bold')

plt.tight_layout()
plt.savefig('category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved as category_revenue.png')

---
## 👥 4. Top Customer Spending

Identify the highest-value customers by total lifetime spending for targeted marketing and retention strategies.

In [ ]:
# Query: Top 15 customers by spending
query_top_customers = """
SELECT
    c.customer_name,
    c.city,
    c.state,
    COUNT(o.order_id) AS total_orders,
    SUM(o.revenue) AS total_spent,
    ROUND(AVG(o.revenue)::numeric, 2) AS avg_order_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_name, c.city, c.state
ORDER BY total_spent DESC
LIMIT 15;
"""

df_top_customers = pd.read_sql(query_top_customers, engine)
df_top_customers

In [ ]:
# Visualization: Customer Spending Bar Chart
fig, ax = plt.subplots(figsize=(16, 8))

colors = sns.color_palette('magma', len(df_top_customers))

bars = ax.barh(
    df_top_customers['customer_name'][::-1],
    df_top_customers['total_spent'][::-1],
    color=colors[::-1],
    edgecolor='white',
    linewidth=0.5
)

# Add value labels with city/state info
for i, (bar, row) in enumerate(zip(bars, df_top_customers[::-1].itertuples())):
    ax.text(
        bar.get_width() + bar.get_width() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'₹{row.total_spent:,.0f}  ({row.city})',
        va='center', fontsize=9, fontweight='bold'
    )

ax.set_xlabel('Total Spending (₹)', fontsize=13)
ax.set_title('Top 15 Customers by Lifetime Spending', fontsize=18, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('customer_spending.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved as customer_spending.png')

---
## 💳 5. Payment Method Distribution

Understand customer payment preferences to optimize the checkout experience.

In [ ]:
# Query: Payment method distribution
query_payment = """
SELECT
    payment_method,
    COUNT(order_id) AS total_orders,
    SUM(revenue) AS total_revenue,
    ROUND(COUNT(order_id) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS usage_pct
FROM orders
GROUP BY payment_method
ORDER BY total_orders DESC;
"""

df_payment = pd.read_sql(query_payment, engine)
df_payment

In [ ]:
# Visualization: Payment Method Donut Chart
fig, ax = plt.subplots(figsize=(10, 8))

colors = sns.color_palette('coolwarm', len(df_payment))

wedges, texts, autotexts = ax.pie(
    df_payment['total_orders'],
    labels=df_payment['payment_method'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=3)
)

for text in autotexts:
    text.set_fontsize(11)
    text.set_fontweight('bold')

# Center label
total_orders = df_payment['total_orders'].sum()
ax.text(0, 0, f'{total_orders:,}\nOrders', ha='center', va='center',
        fontsize=18, fontweight='bold', color='#333')

ax.set_title('Payment Method Distribution', fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('payment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved as payment_distribution.png')

---
## 🔥 6. Category Performance Heatmap

A heatmap view of category performance across multiple metrics for quick comparison.

In [ ]:
# Query: Category performance metrics
query_heatmap = """
SELECT
    p.category,
    SUM(o.revenue) AS revenue,
    COUNT(o.order_id) AS orders,
    SUM(o.quantity) AS units_sold,
    ROUND(AVG(o.revenue)::numeric, 2) AS avg_order_val,
    COUNT(DISTINCT o.customer_id) AS unique_customers
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC;
"""

df_heatmap = pd.read_sql(query_heatmap, engine)
df_heatmap_norm = df_heatmap.set_index('category')

# Normalize each column to 0-1 for comparable heatmap
df_heatmap_norm = (df_heatmap_norm - df_heatmap_norm.min()) / (
    df_heatmap_norm.max() - df_heatmap_norm.min()
)

fig, ax = plt.subplots(figsize=(12, 8))

sns.heatmap(
    df_heatmap_norm,
    annot=df_heatmap.set_index('category').values,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=2,
    linecolor='white',
    cbar_kws={'label': 'Normalized Score'},
    ax=ax
)

ax.set_title('Category Performance Heatmap', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Metrics', fontsize=13)
ax.set_ylabel('Category', fontsize=13)

plt.tight_layout()
plt.savefig('category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved as category_heatmap.png')

---
## 📋 Summary Statistics

In [ ]:
# Overall business metrics
query_summary = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS active_customers,
    COUNT(DISTINCT o.product_id) AS products_sold,
    SUM(o.revenue) AS total_revenue,
    ROUND(AVG(o.revenue)::numeric, 2) AS avg_order_value,
    SUM(o.quantity) AS total_units_sold
FROM orders o;
"""

df_summary = pd.read_sql(query_summary, engine)

print('=' * 50)
print('  BUSINESS SUMMARY')
print('=' * 50)
print(f"  Total Orders:       {df_summary['total_orders'].iloc[0]:>10,}")
print(f"  Active Customers:   {df_summary['active_customers'].iloc[0]:>10,}")
print(f"  Products Sold:      {df_summary['products_sold'].iloc[0]:>10,}")
print(f"  Total Revenue:     ₹{df_summary['total_revenue'].iloc[0]:>12,.2f}")
print(f"  Avg Order Value:   ₹{df_summary['avg_order_value'].iloc[0]:>12,.2f}")
print(f"  Total Units Sold:   {df_summary['total_units_sold'].iloc[0]:>10,}")
print('=' * 50)

In [ ]:
# Close connection
engine.dispose()
print('\n✓ Database connection closed.')
print('✓ All visualizations saved successfully.')